### Testes Unitários

#### Auditoria ETL pt 1

In [6]:
import sys
import os
from pathlib import Path

# 1. Força o Jupyter a enxergar a raiz do projeto (voltando uma pasta)
# Isso permite que ele encontre a pasta 'src'
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Agora os imports vão funcionar perfeitamente
import pandas as pd
import numpy as np
from src.data.make_dataset import clean_carregamento, clean_ramps

# ==========================================
# TESTE 1: Sanidade do Carregamento (H2O e Zeros)
# ==========================================
print("--- INICIANDO TESTE DE CARREGAMENTO ---")
# Como o CWD mudou para o sys.path, vamos garantir que a função ache o arquivo em data/raw/
df_proc = clean_carregamento(path=PROJECT_ROOT / "data" / "raw" / "carregamento_2025.csv")

# 1A. Teste Físico: Nenhum H2O pode ser > 25%
assert df_proc['h2o_pct'].max() <= 25.0, f"FALHA: H2O máximo é {df_proc['h2o_pct'].max()}%"
print("✓ [PASS] Regra de H2O (<=25%) validada.")

# 1B. Teste de Imputação SOTA: Não deve existir NaN no H2O
assert df_proc['h2o_pct'].isna().sum() == 0, "FALHA: Existem valores NaN no H2O"
print("✓ [PASS] Rolling Median imputou todos os nulos com sucesso.")

# ==========================================
# TESTE 2: Geometria Vetorial das RAMPs
# ==========================================
print("\n--- INICIANDO TESTE DE METEOROLOGIA ---")
df_ramps = clean_ramps(path=PROJECT_ROOT / "data" / "raw" / "ramps_2025.csv")

# 2A. Teste Trigonométrico de Pitágoras: u^2 + v^2 deve ser igual a V^2
velocidade_recalculada = np.sqrt(df_ramps['vento_u']**2 + df_ramps['vento_v']**2)
erro_maximo = np.abs(df_ramps['velocidade_vento'] - velocidade_recalculada).max()
assert erro_maximo < 0.01, f"FALHA: A decomposição vetorial quebrou. Erro max: {erro_maximo}"
print(f"✓ [PASS] Identidade de Pitágoras validada (Erro max: {erro_maximo:.5f}).")

print("\n🚀 TODOS OS TESTES PASSARAM! O ETL É SOTA.")

--- INICIANDO TESTE DE CARREGAMENTO ---
✓ [PASS] Regra de H2O (<=25%) validada.
✓ [PASS] Rolling Median imputou todos os nulos com sucesso.

--- INICIANDO TESTE DE METEOROLOGIA ---
✓ [PASS] Identidade de Pitágoras validada (Erro max: 0.00000).

🚀 TODOS OS TESTES PASSARAM! O ETL É SOTA.


#### Auditoria ETL pt 2